In [6]:
%pip install unsloth transformers datasets trl peft accelerate bitsandbytes

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Torch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True
GPU: NVIDIA GeForce GTX 1660


In [2]:
import torch

free, total = torch.cuda.mem_get_info()

print(f"Total VRAM: {total / 1024**3:.2f} GB")
print(f"Free VRAM:  {free / 1024**3:.2f} GB")

Total VRAM: 6.00 GB
Free VRAM:  5.01 GB


In [3]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    device_map="cuda",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


d:\Projects\Healthcare Scheduling System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0809 06:08:24.153000 17172 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.3: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1660. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 355.31it/s]


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

Unsloth 2026.8.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train"
)

{'question': 'What is the Barangay Bagumbayan Health Center?', 'answer': 'The Barangay Bagumbayan Health Center is a government healthcare facility that provides primary healthcare services, health education, disease prevention, and medical consultations to residents of Barangay Bagumbayan, Taguig City.'}


In [ ]:
def format_dolly(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}

dataset = ds.map(format_dolly)

In [7]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="outputs",

    num_train_epochs=2,
    learning_rate=1e-4,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,

    logging_steps=10,
    save_strategy="epoch",

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    optim="adamw_8bit",

    report_to="none",

    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,

    max_length=1024,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Packing train dataset: 100%|██████████| 450/450 [00:00<00:00, 50000.97 examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [8]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9 | Num Epochs = 2 | Total steps = 4
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-2\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs\checkpoint-2.
Unsloth: Restored added_tokens_decoder metadata in outputs\checkpoint-4\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs\checkpoint-4.


TrainOutput(global_step=4, training_loss=2.6693053245544434, metrics={'train_runtime': 1466.1955, 'train_samples_per_second': 0.012, 'train_steps_per_second': 0.003, 'total_flos': 801436114575360.0, 'train_loss': 2.6693053245544434, 'epoch': 2.0})

In [9]:
model.save_pretrained("phi3-dolly-lora")
tokenizer.save_pretrained("phi3-dolly-lora")

Unsloth: Restored added_tokens_decoder metadata in phi3-dolly-lora\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi3-dolly-lora.


('phi3-dolly-lora\\tokenizer_config.json',
 'phi3-dolly-lora\\chat_template.jinja',
 'phi3-dolly-lora\\tokenizer.json')

In [10]:
model.save_pretrained_gguf(
    "gguf_model",
    tokenizer,
    quantization_method="q4_k_m",
)

Unsloth: Merging model weights to 16-bit format...
Unsloth: Not enough free disk to keep `unsloth/phi-3-mini-4k-instruct` in the Hugging Face cache (need ~14.2GB free, have 3.7GB). Downloading straight to the merge directory instead; the next export will re-download it.


Unsloth: Restored added_tokens_decoder metadata in gguf_model\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gguf_model.


Found HuggingFace hub cache directory: C:\Users\user\.cache\huggingface\hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  1.72it/s]


Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:17<00:00, 98.65s/it] 


Unsloth: Merge process complete. Saved to `d:\Projects\Healthcare Scheduling System\ai\notebooks\gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10327-mix-b1e9972 (app-b10327-mix-b1e9972-windows-x64-cpu.zip) - skipping compilation.


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Converter dependency install failed ([FAIL] Command `"d:\Projects\Healthcare Scheduling System\.venv\Scripts\python.exe" -m pip install gguf protobuf sentencepiece mistral_common` failed with error `WARNING: Failed to remove contents in a temporary directory 'D:\Projects\Healthcare Scheduling System\.venv\Lib\site-packages\~umpy.libs'.`
); conversion self-heals if needed.


Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_model_gguf\\phi-3-mini-4k-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['gguf_model_gguf\\phi-3-mini-4k-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: C:\Users\user\.unsloth\llama.cpp\build\bin\Release\llama-cli.exe --model gguf_model_gguf\phi-3-mini-4k-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to gguf_model_gguf\Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f gguf_model_gguf\Modelfile


{'save_directory': 'gguf_model',
 'gguf_directory': 'gguf_model_gguf',
 'gguf_files': ['gguf_model_gguf\\phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'gguf_model_gguf\\Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}